In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configs


In [47]:
DECISION_THRESHOLD = 0.5 #default = 0.5
DISPLAY_PLOTS = False

# Datasets

In [48]:
FEATURE_SET = "struct"  # options: "original, struct"

DATASET_PATHS = {
    "original": {
        "main": "Data/features_main.csv",
        "temporal": "Data/features_test_temporal.csv",
        "obfuscated": "Data/features_test_obfuscated.csv",
        "notinject": "Data/features_test_notinject.csv",
    },
    "struct": {
        "main": "Data/features_main_struct.csv",
        "temporal": "Data/features_test_temporal_struct.csv",
        "obfuscated": "Data/features_test_obfuscated_struct.csv",
        "notinject": "Data/features_test_notinject_struct.csv",
    },
}

paths = DATASET_PATHS[FEATURE_SET]
df = pd.read_csv(paths["main"], keep_default_na=False)
temporal_df = pd.read_csv(paths["temporal"], keep_default_na=False)
obfuscated_df = pd.read_csv(paths["obfuscated"], keep_default_na=False)
notinject_df = pd.read_csv(paths["notinject"], keep_default_na=False)

print(f"Loaded feature set: '{FEATURE_SET}'")
print("Split values found in main dataset:", df['split'].unique())
print("Shape:", df.shape)

Loaded feature set: 'struct'
Split values found in main dataset: <StringArray>
['train', 'val', 'test']
Length: 3, dtype: str
Shape: (13489, 33)


# Random Forest

In [49]:
from sklearn.ensemble import RandomForestClassifier

In [50]:
id_cols = ['uid', 'split', 'label', 'source', 'attack_type']
feature_cols = [c for c in df.columns if c not in id_cols]

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

In [51]:
train_uids = set(train_df['uid'])
eval_sets = {
    'val': val_df,
    'test': test_df,
    'temporal': temporal_df,
    'obfuscated': obfuscated_df,
    'notinject': notinject_df,
}
for name, edf in eval_sets.items():
    overlap = train_uids & set(edf['uid'])
    if overlap:
        print(f"WARNING: {len(overlap)} uid(s) shared between train and {name} — check for leakage")

In [52]:
X_train, y_train = train_df[feature_cols], train_df['label']
X_val, y_val = val_df[feature_cols], val_df['label']

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total 

In [53]:
def evaluate(df_test, name, threshold=DECISION_THRESHOLD, plot=DISPLAY_PLOTS):
    X = df_test[feature_cols]
    y = df_test['label']

    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)

    print(f"\n--- {name} (n={len(y)}, threshold={threshold}) ---")

    if y.nunique() == 1:
        only_class = y.iloc[0]
        if only_class == 0:
            fpr = (preds == 1).mean()
            print(f"All-benign set. False Positive Rate: {fpr:.4f}")
        else:
            fnr = (preds == 0).mean()
            print(f"All-attack set. False Negative Rate: {fnr:.4f}  |  Recall: {1 - fnr:.4f}")
    else:
        print("Accuracy:", accuracy_score(y, preds))
        print(classification_report(y, preds))

    if plot:
        cm = confusion_matrix(y, preds, labels=[0, 1])
        plt.figure()
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Predicted Benign', 'Predicted Attack'],
                    yticklabels=['Actual Benign', 'Actual Attack'])
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.title(f'Confusion Matrix — {name}')
        plt.show()

    return df_test, preds, probs


In [54]:
train_preds = model.predict(X_train)
print("--- Training Set ---")
print("Accuracy:", accuracy_score(y_train, train_preds))
print(classification_report(y_train, train_preds))

--- Training Set ---
Accuracy: 0.9456093003944364
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      6052
           1       0.98      0.87      0.92      3582

    accuracy                           0.95      9634
   macro avg       0.95      0.93      0.94      9634
weighted avg       0.95      0.95      0.94      9634



In [55]:
Val_result = evaluate(val_df, 'Validation')


--- Validation (n=1927, threshold=0.5) ---
Accuracy: 0.9190451478982875
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      1210
           1       0.95      0.82      0.88       717

    accuracy                           0.92      1927
   macro avg       0.93      0.90      0.91      1927
weighted avg       0.92      0.92      0.92      1927



In [56]:
if not test_df.empty:
    evaluate(test_df, 'Test (main)')


--- Test (main) (n=1928, threshold=0.5) ---
Accuracy: 0.9092323651452282
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      1211
           1       0.94      0.80      0.87       717

    accuracy                           0.91      1928
   macro avg       0.92      0.89      0.90      1928
weighted avg       0.91      0.91      0.91      1928



# Targeted stress tests

In [57]:
temp_df, temp_preds, temp_probs = evaluate(temporal_df, 'Temporal')
obf_df, obf_preds, obf_probs = evaluate(obfuscated_df, 'Obfuscated')
notinj_df, notinj_preds, notinj_probs = evaluate(notinject_df, 'Notinject')


--- Temporal (n=485, threshold=0.5) ---
Accuracy: 0.6948453608247422
              precision    recall  f1-score   support

           0       0.73      0.82      0.77       308
           1       0.61      0.47      0.53       177

    accuracy                           0.69       485
   macro avg       0.67      0.65      0.65       485
weighted avg       0.68      0.69      0.68       485


--- Obfuscated (n=1800, threshold=0.5) ---
Accuracy: 0.8561111111111112
              precision    recall  f1-score   support

           0       0.78      0.98      0.87       900
           1       0.98      0.73      0.84       900

    accuracy                           0.86      1800
   macro avg       0.88      0.86      0.85      1800
weighted avg       0.88      0.86      0.85      1800


--- Notinject (n=339, threshold=0.5) ---
All-benign set. False Positive Rate: 0.1180


In [58]:
combined = pd.concat([temporal_df, obfuscated_df, notinject_df])
combined_result = evaluate(combined, 'Combined targeted stress tests (temporal + obfuscated + notinject)')


--- Combined targeted stress tests (temporal + obfuscated + notinject) (n=2624, threshold=0.5) ---
Accuracy: 0.8296493902439024
              precision    recall  f1-score   support

           0       0.81      0.93      0.87      1547
           1       0.87      0.69      0.77      1077

    accuracy                           0.83      2624
   macro avg       0.84      0.81      0.82      2624
weighted avg       0.84      0.83      0.83      2624



# Logistic Regression

In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [60]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [61]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [62]:
def evaluate_lr(df_test, name, threshold=DECISION_THRESHOLD, plot=DISPLAY_PLOTS):
    X = scaler.transform(df_test[feature_cols])
    y = df_test['label']

    probs = lr_model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)

    print(f"\n--- {name} [Logistic Regression] (n={len(y)}, threshold={threshold}) ---")

    if y.nunique() == 1:
        only_class = y.iloc[0]
        if only_class == 0:
            fpr = (preds == 1).mean()
            print(f"All-benign set. False Positive Rate: {fpr:.4f}")
        else:
            fnr = (preds == 0).mean()
            print(f"All-attack set. False Negative Rate: {fnr:.4f}  |  Recall: {1 - fnr:.4f}")
    else:
        print("Accuracy:", accuracy_score(y, preds))
        print(classification_report(y, preds))

    if plot:
        cm = confusion_matrix(y, preds, labels=[0, 1])
        plt.figure()
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Predicted Benign', 'Predicted Attack'],
                    yticklabels=['Actual Benign', 'Actual Attack'])
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.title(f'Confusion Matrix — {name} [Logistic Regression]')
        plt.show()

    return df_test, preds, probs

In [63]:
train_preds_lr = lr_model.predict(X_train_scaled)
print("--- Training Set [Logistic Regression] ---")
print("Accuracy:", accuracy_score(y_train, train_preds_lr))
print(classification_report(y_train, train_preds_lr))

--- Training Set [Logistic Regression] ---
Accuracy: 0.8916337969690679
              precision    recall  f1-score   support

           0       0.90      0.93      0.91      6052
           1       0.87      0.84      0.85      3582

    accuracy                           0.89      9634
   macro avg       0.89      0.88      0.88      9634
weighted avg       0.89      0.89      0.89      9634



In [64]:
evaluate_lr(val_df, 'Validation')
if not test_df.empty:
    evaluate_lr(test_df, 'Test (main)')


--- Validation [Logistic Regression] (n=1927, threshold=0.5) ---
Accuracy: 0.8780487804878049
              precision    recall  f1-score   support

           0       0.89      0.92      0.90      1210
           1       0.86      0.80      0.83       717

    accuracy                           0.88      1927
   macro avg       0.87      0.86      0.87      1927
weighted avg       0.88      0.88      0.88      1927


--- Test (main) [Logistic Regression] (n=1928, threshold=0.5) ---
Accuracy: 0.8843360995850622
              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1211
           1       0.85      0.83      0.84       717

    accuracy                           0.88      1928
   macro avg       0.88      0.87      0.88      1928
weighted avg       0.88      0.88      0.88      1928



# Targeted stress tests

In [65]:
temp_df_lr, temp_preds_lr, temp_probs_lr = evaluate_lr(temporal_df, 'Temporal')
obf_df_lr, obf_preds_lr, obf_probs_lr = evaluate_lr(obfuscated_df, 'Obfuscated')
notinj_df_lr, notinj_preds_lr, notinj_probs_lr = evaluate_lr(notinject_df, 'Notinject')


--- Temporal [Logistic Regression] (n=485, threshold=0.5) ---
Accuracy: 0.6515463917525773
              precision    recall  f1-score   support

           0       0.70      0.79      0.74       308
           1       0.53      0.42      0.47       177

    accuracy                           0.65       485
   macro avg       0.62      0.60      0.60       485
weighted avg       0.64      0.65      0.64       485


--- Obfuscated [Logistic Regression] (n=1800, threshold=0.5) ---
Accuracy: 0.585
              precision    recall  f1-score   support

           0       0.66      0.36      0.46       900
           1       0.56      0.81      0.66       900

    accuracy                           0.58      1800
   macro avg       0.61      0.58      0.56      1800
weighted avg       0.61      0.58      0.56      1800


--- Notinject [Logistic Regression] (n=339, threshold=0.5) ---
All-benign set. False Positive Rate: 0.3717


In [66]:
combined_lr_df, combined_lr_preds, combined_lr_probs = evaluate_lr(
    combined, 'Combined stress tests (temporal + obfuscated + notinject)'
)


--- Combined stress tests (temporal + obfuscated + notinject) [Logistic Regression] (n=2624, threshold=0.5) ---
Accuracy: 0.6028963414634146
              precision    recall  f1-score   support

           0       0.74      0.50      0.60      1547
           1       0.51      0.75      0.61      1077

    accuracy                           0.60      2624
   macro avg       0.63      0.62      0.60      2624
weighted avg       0.65      0.60      0.60      2624



## Saving

In [67]:
import os
import joblib
from sklearn.pipeline import Pipeline

os.makedirs("models", exist_ok=True)

# Random Forest
joblib.dump(model, "models/rf_model.joblib")

# Logistic Regression
lr_pipeline = Pipeline([
    ('scaler', scaler),
    ('clf', lr_model)
])
joblib.dump(lr_pipeline, "models/lr_pipeline.joblib")

# Shared: exact column order/names both models expect at inference time
joblib.dump(feature_cols, "models/feature_cols.joblib")

print("Saved to models/:")
print(os.listdir("models"))

Saved to models/:
['feature_cols.joblib', 'lr_pipeline.joblib', 'rf_model.joblib']
